# 第 03 章练习

完成下面的练习，结合自己的运行结果作答。


<a id="s-03-06_chapter_test"></a>


## 3.6 MPI 进程扩展性

比较不同进程数下的计算、通信和总时间。


### 检查 MPI 工具

下面查找 cmake、mpicxx 和 mpirun。若缺少工具，先安装或加载 MPI 环境，再继续。

In [ ]:
%%bash
set -e
cmake --version
mpicxx --version
mpirun --version


### 比较时保持什么不变

本节只改变进程数，矩阵、预热和重复次数保持一致。使用已有程序，无需修改源码。

### 运行并记录结果

执行下面的命令，保存本次输出。

In [ ]:
%%bash
set -eo pipefail
cd src/mpi_spmv
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CXX_COMPILER=mpicxx
cmake --build build -j
mkdir -p results
# 学生修改：下面的进程数列表，勿超过已分配的 CPU 资源。
for processes in 1 2 4; do
  mpirun -np "$processes" ./build/spmv_mpi \
    --matrix U1 --warmup 1 --repeat 3 \
    --csv "results/U1_${processes}p.csv" 2>&1 | tee "results/U1_${processes}p.log"
done


### 练习与提交

1. 假设按行均分，手算 10 行分给 4 个 rank 的行区间；说明分到 0 行时为什么仍要参加 collective。
2. 按当前 CPU slot 选择 1/2/4 rank，查看主 CSV 和 `.per_rank.csv`。确认绝对 L2 误差小于 `1e-6`。
3. 用每个 rank 的 nnz 和平均 compute 时间计算负载差异。当前程序按 nnz 分区；再手算同一矩阵按行均分的边界，解释两者的区别。
4. 分别列出一次性 distribution 与每轮 total。比较 broadcast、compute、gather，提出一个进程数单变量实验。

不要把一次性分发加进程序已经输出的每轮 total；各阶段 MAX 也不要求相加等于 total。

以本册四进程结果为例，各 rank 的平均 compute 最大为 0.215097 ms、平均约 0.214556 ms，算得 `load_imbalance≈0.00252`。


[参考答案](answer/03.06_answer.md)
